In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import joblib
import matplotlib.pyplot as plt

plt.style.use('dark_background')

In [2]:
# 1. Generate "Normal" usage (e.g., normal appliances running)
np.random.seed(42)
normal_draw = np.random.normal(loc=1500, scale=800, size=5000)
# Ensure no negative power
normal_draw = np.clip(normal_draw, 100, 4500) 

# 2. Generate "Anomalies" (e.g., short circuits, all appliances on at once)
anomalies_high = np.random.uniform(low=6000, high=9000, size=50)
anomalies_low = np.random.uniform(low=0, high=50, size=5) # Almost 0 draw

# Combine them into a dataset
all_data = np.concatenate([normal_draw, anomalies_high, anomalies_low])
df = pd.DataFrame(all_data, columns=['power_watts'])

print(f"Generated {len(df)} hours of power readings.")

Generated 5055 hours of power readings.


In [3]:
print("🧠 Training Isolation Forest...")

# contamination=0.01 means we expect about 1% of our data to be anomalies
model = IsolationForest(contamination=0.01, random_state=42)

# Train the model (Note: It expects a 2D array, so we use df[['power_watts']])
model.fit(df[['power_watts']])

# Let's test it immediately!
test_values = pd.DataFrame([[150], [2000], [4000], [7500]], columns=['power_watts'])
predictions = model.predict(test_values)

# Isolation Forest returns 1 for Normal, and -1 for Anomaly
for i, watts in enumerate(test_values['power_watts']):
    status = "🚨 ANOMALY" if predictions[i] == -1 else "✅ Normal"
    print(f"Power Draw: {watts}W -> {status}")

🧠 Training Isolation Forest...
Power Draw: 150W -> ✅ Normal
Power Draw: 2000W -> ✅ Normal
Power Draw: 4000W -> ✅ Normal
Power Draw: 7500W -> 🚨 ANOMALY


In [4]:
# Save the brain to our backend store
model_path = "../backend/model_store/isolation_forest_v1.pkl"
joblib.dump(model, model_path)

print(f"✅ Anomaly Detector serialized and saved to: {model_path}")

✅ Anomaly Detector serialized and saved to: ../backend/model_store/isolation_forest_v1.pkl
